In [ ]:
import torch
from decoding_core.models import CoupledDynamixRegressor, train_coupled
from decoding_core.utils_finetune import load_finetuning_data
from dynamix.utilities.utilities import load_hf_model

In [2]:
FOLDER = "/home/sgurbuz/nasShare/projects/sgurbuz/dynamix_tryout/data_1d_vxlbl/ALCIS_155_macroREF"
train_dl, test_dl, y_mean, y_std, modul_list, test_probe = load_finetuning_data(
    path_folder=FOLDER,
    test_probe=None,        # LOPO; or set probe_id=... for single-probe
    batch_size=8,
)

Found 28 voltammogram folders
Total sweeps: 127050  | voltammogram shape: (1000, 1)
Available probes (7): ['ALCIS_155_macroREF__ALC1_INM001_01bW04R01M'
 'ALCIS_155_macroREF__ALC2_INM001_02bW02R01M'
 'ALCIS_155_macroREF__ALC3_INM001_03bW02R01M'
 'ALCIS_155_macroREF__ALC3_INM001_03bW06R02M'
 'ALCIS_155_macroREF__ALC4_INM001_04bW02R01M'
 'ALCIS_155_macroREF__ALC4_INM001_04bW06R02M'
 'ALCIS_155_macroREF__ALC4_INM001_04bW08R03M']
No test_probe given — randomly selected: ALCIS_155_macroREF__ALC1_INM001_01bW04R01M
[LOPO] test probe = ALCIS_155_macroREF__ALC1_INM001_01bW04R01M
Train sweeps: 108900  | Test sweeps: 18150
y_mean: [461.6529    465.12396   461.6529      7.3948784]  y_std: [7.2022064e+02 7.2429999e+02 7.2008380e+02 7.0613898e-02]


In [3]:

dynamix_model = load_hf_model("dynamix-3d-alrnn-v1.0")

In [4]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'



model = CoupledDynamixRegressor(dynamix_model, horizon=100, n_neuromodul=4)

optimizer = torch.optim.Adam([
    {'params': model.gating_params,            'lr': 1e-4},
    {'params': list(model.head.parameters()),  'lr': 5e-4},
])
loss_fn = torch.nn.MSELoss()

# sanity check: only gating MLP weights + head should be trainable
trainable = [n for n, p in model.named_parameters() if p.requires_grad]
print("Trainable:", trainable)

train_losses, val_losses = train_coupled(
    model, train_dl, test_dl, loss_fn, optimizer,
    device=device, num_epochs=100)

Trainable: ['dynamix.gating_network.mlp_layer1.weight', 'dynamix.gating_network.mlp_layer2.weight', 'head.conv.0.weight', 'head.conv.0.bias', 'head.conv.3.weight', 'head.conv.3.bias', 'head.fc.0.weight', 'head.fc.0.bias']

GRADIENT ISOLATION CHECK
Trainable params (8):
    + dynamix.gating_network.mlp_layer1.weight
    + dynamix.gating_network.mlp_layer2.weight
    + head.conv.0.weight
    + head.conv.0.bias
    + head.conv.3.weight
    + head.conv.3.bias
    + head.fc.0.weight
    + head.fc.0.bias
Frozen (sample): ['dynamix.B', 'dynamix.experts.0.A', 'dynamix.experts.0.W'] ... [+Ellipsis]
Trainable: 10,034 / 19,860  (50.52%)

Starting finetuning: 100 epochs | 108900 train / 18150 val sweeps | horizon=100 | device=cuda

  [first-step grad check]
    mlp_layer1.weight.grad : OK norm=1.096e-02
    mlp_layer2.weight.grad : OK norm=7.513e-03
    expert[0] param.grad   : None  (OK, frozen)
    conv.weight.grad       : None  (OK, frozen)

  ep   1 | batch   1/13613 | batch loss 1.9309
  ep  

KeyboardInterrupt: 